# Notebook 2 — Measurement, Observables & Quantum Simulation

**Qiskit Fall Fest 2026 — University of Ottawa**

**Difficulty:** Beginner → Intermediate
**Estimated time:** 100–120 minutes
**Prerequisites:** Notebook 1 (circuits, gates, `Statevector`, visualization).

## Learning objectives
By the end of this notebook you will be able to:

- ✓ explain the difference between a quantum state, measurement, shots, counts, and probabilities
- ✓ simulate circuits exactly (statevector) and via sampling (shots)
- ✓ measure in the X, Y, and Z bases
- ✓ construct Pauli observables with `SparsePauliOp`
- ✓ calculate expectation values by hand and with Qiskit
- ✓ use the modern `Sampler` and `Estimator` primitives
- ✓ show, with real numbers, why entanglement produces correlations invisible to either qubit alone

> **Note on scope.** This notebook uses only exact and shot-based **simulation**. Noise models are introduced in Notebook 3, and real IBM hardware execution is covered in Notebook 4.

## 1. From Circuit to Classical Information

$$\text{Circuit} \;\rightarrow\; \text{Quantum state} \;\rightarrow\; \text{Measurement} \;\rightarrow\; \text{Classical information}$$

A quantum computer's state lives in a Hilbert space that we, as external classical observers, cannot read directly. The *only* way to extract information is measurement, which:

1. Collapses the state to a single basis outcome (probabilistically).
2. Reports that outcome as a classical bit (0 or 1) per measured qubit.
3. Destroys the original superposition — you cannot "measure again" and get the pre-measurement state back.

This is why, on **real hardware**, you can only ever get *measurement statistics* (counts), never the exact statevector. `Statevector` (used throughout Notebook 1) is a simulator-only tool that "peeks" at the exact quantum state — impossible on physical devices.

## 2. Amplitudes and Probabilities

For a single qubit $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$:

$$P(0) = |\alpha|^2 \qquad P(1) = |\beta|^2$$

Let's verify this numerically for a few states.

In [ ]:
import qiskit, qiskit_aer
print("Qiskit:", qiskit.__version__, "| Aer:", qiskit_aer.__version__)

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np

qc_plus = QuantumCircuit(1)
qc_plus.h(0)
sv = Statevector(qc_plus)
print("Amplitudes:", sv.data)
print("Probabilities:", sv.probabilities_dict())

> **Checkpoint.** For $|+\rangle$, both amplitudes are $\tfrac{1}{\sqrt2} \approx 0.707$. Confirm $P(0) = P(1) = |0.707|^2 \approx 0.5$ matches the printed probabilities.

## 3. Ideal Statevector Simulation vs. Shot-Based Sampling

`Statevector` gives the **exact** mathematical answer instantly — no randomness, no shots. Real measurement (and even simulated measurement) is fundamentally different: each individual "shot" produces one random outcome drawn according to the state's probabilities. Only *many* shots let you estimate those probabilities.

In [ ]:
from qiskit.primitives import StatevectorSampler

qc_meas = QuantumCircuit(1, 1)
qc_meas.h(0)
qc_meas.measure(0, 0)

sampler = StatevectorSampler()
job = sampler.run([qc_meas], shots=10)
result = job.result()
counts = result[0].data.c.get_counts()
print("10 shots:", counts)

We just used the **`StatevectorSampler`** — Qiskit's exact-simulation implementation of the modern `SamplerV2` primitive interface. Note that with only 10 shots, the counts will likely *not* be a perfect 50/50 split — that's expected statistical noise, not a bug.

## 4. Measurement in Qiskit

In [ ]:
qc_full_meas = QuantumCircuit(2)
qc_full_meas.h(0)
qc_full_meas.cx(0, 1)
qc_full_meas.measure_all()   # adds a NEW classical register called 'meas'
qc_full_meas.draw("mpl")

> **Try it yourself.** What classical register name does `measure_all()` create? Check `qc_full_meas.cregs`.

In [ ]:
print(qc_full_meas.cregs)

### Convergence with Increasing Shots

> **What do you expect?** As shot count increases from 10 to 10,000, do you expect the measured counts to converge toward, or drift away from, the ideal 50/50 split for $|+\rangle$?

In [ ]:
qc_plus_meas = QuantumCircuit(1, 1)
qc_plus_meas.h(0)
qc_plus_meas.measure(0, 0)

sampler = StatevectorSampler()
for shots in [10, 100, 1_000, 10_000]:
    job = sampler.run([qc_plus_meas], shots=shots)
    counts = job.result()[0].data.c.get_counts()
    p0 = counts.get('0', 0) / shots
    print(f"shots={shots:>6} -> counts={counts}  P(0) estimate={p0:.3f}")

As shots increase, the empirical fraction converges toward the true probability $P(0)=0.5$ — this is the **law of large numbers** in action, and it's exactly why real quantum hardware jobs are run with many shots (typically hundreds to thousands) rather than just one.

## 5. Histograms and Result Visualization

In [ ]:
from qiskit.visualization import plot_histogram

job = sampler.run([qc_plus_meas], shots=1000)
counts = job.result()[0].data.c.get_counts()
plot_histogram(counts)

In [ ]:
job_a = sampler.run([qc_plus_meas], shots=200)
job_b = sampler.run([qc_plus_meas], shots=200)
counts_a = job_a.result()[0].data.c.get_counts()
counts_b = job_b.result()[0].data.c.get_counts()

plot_histogram([counts_a, counts_b], legend=["run A", "run B"])

`plot_histogram` can accept a **list** of counts dictionaries to compare multiple runs side by side — useful later for comparing ideal vs. noisy vs. hardware results (Notebooks 3–4).

## 6. Measurement Basis

By default, `qc.measure()` measures in the **Z basis** (computational basis: $|0\rangle$, $|1\rangle$). To measure in a different basis, we *rotate the state* so that the desired basis aligns with Z, then measure normally.

- **Z-basis measurement** — direct `qc.measure(...)`, no rotation needed.
- **X-basis measurement** — apply `H`, then measure in Z. ($H$ maps $|+\rangle \to |0\rangle$ and $|-\rangle \to |1\rangle$.)
- **Y-basis measurement** — apply `S†` then `H`, then measure in Z.

In [ ]:
def measure_in_basis(qc, qubit, clbit, basis):
    if basis == "Z":
        pass
    elif basis == "X":
        qc.h(qubit)
    elif basis == "Y":
        qc.sdg(qubit)
        qc.h(qubit)
    else:
        raise ValueError("basis must be 'X', 'Y', or 'Z'")
    qc.measure(qubit, clbit)

qc_plus_x = QuantumCircuit(1, 1)
qc_plus_x.h(0)                       # prepare |+>
measure_in_basis(qc_plus_x, 0, 0, "X")
counts_x = sampler.run([qc_plus_x], shots=1000).result()[0].data.c.get_counts()
print("Measuring |+> in the X basis:", counts_x)

> **Checkpoint.** We prepared $|+\rangle$ and measured in the X basis. Since $|+\rangle$ *is* the "+1 eigenstate" of X, we should see almost all outcomes as `0` (by convention, `0` corresponds to the `+1` eigenvalue after the basis-change rotation). Confirm this matches the printed counts.

In [ ]:
qc_zero_y = QuantumCircuit(1, 1)   # |0> measured in Y basis — should be 50/50
measure_in_basis(qc_zero_y, 0, 0, "Y")
counts_y = sampler.run([qc_zero_y], shots=1000).result()[0].data.c.get_counts()
print("Measuring |0> in the Y basis:", counts_y)

## 7. Observables

An **observable** is a mathematical object representing something we can measure — mathematically, a Hermitian operator. Its **expectation value** $\langle O \rangle = \langle\psi|O|\psi\rangle$ is the *average* outcome we would get if we measured that observable many times.

For a single qubit, the Pauli operators $I, X, Y, Z$ form a natural basis for observables. Their expectation values $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ are exactly the three Cartesian coordinates of the qubit's **Bloch vector** — this is why the Bloch sphere and Pauli observables are two views of the same underlying information.

## 8. SparsePauliOp

In [ ]:
from qiskit.quantum_info import SparsePauliOp

op_x = SparsePauliOp("X")
op_z = SparsePauliOp("Z")
op_zz = SparsePauliOp("ZZ")
op_xx = SparsePauliOp("XX")

for op in [op_x, op_z, op_zz, op_xx]:
    print(op)

**Qubit ordering in Pauli strings matches the same little-endian convention from Notebook 1**: in `"ZI"`, the rightmost `Z` acts on qubit 0 and the leftmost `I` acts on qubit 1. So `"ZI"` means "measure $Z$ on qubit 0, do nothing on qubit 1" — *not* the reverse.

In [ ]:
labels = ["XI", "IX", "ZI", "IZ", "ZZ", "XX", "YY", "XZ", "ZX"]
for label in labels:
    print(label, "->", SparsePauliOp(label))

> **Common mistake.** Reading `"XZ"` as "X on qubit 1, Z on qubit 0" is backwards for some libraries — but matches Qiskit's own little-endian convention if you keep in mind position 0 (rightmost) = qubit 0.

## 9. Expectation Values

> **Checkpoint — predict before running.** For the states $|0\rangle$, $|1\rangle$, $|+\rangle$, $|-\rangle$, predict $\langle Z\rangle$ and $\langle X\rangle$:
> - $|0\rangle$: $\langle Z\rangle = ?$, $\langle X\rangle = ?$
> - $|1\rangle$: $\langle Z\rangle = ?$, $\langle X\rangle = ?$
> - $|+\rangle$: $\langle Z\rangle = ?$, $\langle X\rangle = ?$
> - $|-\rangle$: $\langle Z\rangle = ?$, $\langle X\rangle = ?$

In [ ]:
states = {
    "|0>": QuantumCircuit(1),
    "|1>": QuantumCircuit(1),
    "|+>": QuantumCircuit(1),
    "|->": QuantumCircuit(1),
}
states["|1>"].x(0)
states["|+>"].h(0)
states["|->"].x(0); states["|->"].h(0)

for name, circ in states.items():
    sv = Statevector(circ)
    z_exp = sv.expectation_value(SparsePauliOp("Z")).real
    x_exp = sv.expectation_value(SparsePauliOp("X")).real
    print(f"{name}: <Z> = {z_exp:+.2f}   <X> = {x_exp:+.2f}")

`Statevector.expectation_value(observable)` computes $\langle\psi|O|\psi\rangle$ **exactly** — this is the noise-free, shot-free mathematical answer, analogous to how `Statevector` itself gives the exact state.

## 10. Sampler vs. Estimator

$$\text{Sampler} \;\rightarrow\; \text{samples / bitstrings / measurement statistics}$$
$$\text{Estimator} \;\rightarrow\; \text{expectation values of observables}$$

- Use **Sampler** when you want raw measurement outcomes (counts, bitstrings) — the circuit must contain measurements.
- Use **Estimator** when you want expectation values of specific observables directly — the circuit should generally **not** contain measurements; you instead supply the observable(s) alongside the circuit.

Both come in modern **V2** forms. This course uses only V2 — the older V1 primitives (`Sampler`, `Estimator` without the "V2" suffix from early Qiskit Runtime versions) are deprecated and removed from current `qiskit-ibm-runtime` releases.

In [ ]:
from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator()

bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)

zz = SparsePauliOp("ZZ")
job = estimator.run([(bell, zz)])
result = job.result()
print("Estimator <ZZ> for Bell state:", result[0].data.evs)

This is the **Primitive Unified Bloc (PUB)** pattern: each entry in the list passed to `.run()` is a tuple of `(circuit, observable[, parameter_values][, precision])`. We'll use this exact pattern again in Notebooks 3 and 4 with noisy simulators and real hardware — only the *object* backing the Estimator changes, not the calling pattern.

## 11. Entanglement Through Observables

> **Checkpoint — predict before running.** For the Bell state $\tfrac{1}{\sqrt2}(|00\rangle+|11\rangle)$, predict: $\langle ZI\rangle$, $\langle IZ\rangle$, $\langle ZZ\rangle$, $\langle XX\rangle$. *(Hint: think about whether each individual qubit looks "random" on its own, and whether the two qubits' outcomes are correlated.)*

In [ ]:
bell_sv = Statevector(bell)

observables = ["ZI", "IZ", "ZZ", "XX"]
for label in observables:
    val = bell_sv.expectation_value(SparsePauliOp(label)).real
    print(f"<{label}> = {val:+.3f}")

**Key learning moment.** $\langle ZI\rangle = \langle IZ\rangle = 0$: each qubit *individually* looks completely random (50/50) — knowing nothing about qubit 1 tells you nothing about its own average. But $\langle ZZ\rangle = 1$ and $\langle XX\rangle = 1$: the *joint* observable is perfectly correlated. This is the signature of entanglement — **individual qubits can appear random while their measurements are strongly correlated with each other.** No classical system of independent coins can reproduce this combination of individually-random-but-jointly-correlated behaviour.

In [ ]:
product_state = QuantumCircuit(2)
product_state.h(0)
product_state.h(1)   # two INDEPENDENT qubits in |+>, no entangling gate
product_sv = Statevector(product_state)

print("Product state (no entanglement):")
for label in observables:
    val = product_sv.expectation_value(SparsePauliOp(label)).real
    print(f"  <{label}> = {val:+.3f}")

Contrast this with the Bell state: for the unentangled product state, $\langle ZZ\rangle = 0$ (no correlation) even though $\langle XX\rangle = 1$ here (a coincidence of this particular product state — try changing one `h` to nothing and see $\langle XX\rangle$ change too). The Bell state is special because it shows *strong correlation in multiple complementary bases simultaneously* — something no product state can replicate.

## 12. Exercises

**Exercise 1 — Try it yourself.** Measure $|+\rangle$ with 5, 50, and 5000 shots. Watch the estimated probability converge.

**Exercise 2 — Exercise.** Measure $|+\rangle$ in the X basis (reuse `measure_in_basis`). What outcome distribution do you expect, and does the result match?

**Exercise 3 — Exercise.** Calculate $\langle Z\rangle$ for $|0\rangle$ and $|1\rangle$ using `StatevectorEstimator` (not `Statevector.expectation_value`) to practice the primitive-based workflow.

**Exercise 4 — Exercise.** Calculate $\langle X\rangle$ for $|+\rangle$ using `StatevectorEstimator`.

**Exercise 5 — Exercise.** Calculate $\langle ZZ\rangle$ for a Bell state using `StatevectorEstimator` and confirm it matches the `Statevector.expectation_value` result from Section 11.

**Exercise 6 — Challenge.** Build a 2-qubit product state (no entangling gate) and a Bell state. Compute $\langle ZZ\rangle$ and $\langle XX\rangle$ for both and explain the differences in your own words.

**Exercise 7 — Challenge.** Create your own `SparsePauliOp` for a 3-qubit system (e.g. `"ZZI"` or `"XYZ"`) and calculate its expectation value for the 3-qubit GHZ state from Notebook 1.

### Solutions (collapsed — try the exercises first!)

```python
# Exercise 1
for shots in [5, 50, 5000]:
    c = sampler.run([qc_plus_meas], shots=shots).result()[0].data.c.get_counts()
    print(shots, c)

# Exercise 2
qc_x_basis = QuantumCircuit(1, 1)
qc_x_basis.h(0)
measure_in_basis(qc_x_basis, 0, 0, "X")
print(sampler.run([qc_x_basis], shots=1000).result()[0].data.c.get_counts())

# Exercise 3
for name, circ in [("|0>", QuantumCircuit(1)), ("|1>", states["|1>"])]:
    r = estimator.run([(circ, SparsePauliOp("Z"))]).result()
    print(name, r[0].data.evs)

# Exercise 4
r = estimator.run([(states["|+>"], SparsePauliOp("X"))]).result()
print(r[0].data.evs)

# Exercise 5
r = estimator.run([(bell, SparsePauliOp("ZZ"))]).result()
print(r[0].data.evs)

# Exercise 6 — see Section 11 code cells directly

# Exercise 7
ghz3 = QuantumCircuit(3); ghz3.h(0); ghz3.cx(0,1); ghz3.cx(1,2)
val = Statevector(ghz3).expectation_value(SparsePauliOp("ZZI")).real
print(val)
```

## Qiskit Cheat Sheet — Notebook 2

| Task | Code |
|---|---|
| Observable | `SparsePauliOp("ZZ")` |
| Exact expectation value | `Statevector(qc).expectation_value(observable)` |
| Sampler (shots) | `StatevectorSampler().run([qc], shots=n)` |
| Get counts | `result[0].data.<creg_name>.get_counts()` |
| Estimator (expectation values) | `StatevectorEstimator().run([(qc, observable)])` |
| Get expectation values | `result[0].data.evs` |
| Histogram | `plot_histogram(counts)` or `plot_histogram([counts_a, counts_b])` |
| Measure basis change | `H` for X-basis, `S† then H` for Y-basis, before `measure()` |

## Common Mistakes

- **Confusing shots with number of qubits.** Shots = number of repeated circuit executions; qubits = size of the quantum register. Unrelated quantities.
- **Forgetting measurement.** An `Estimator` circuit should generally have *no* measurements — observables are computed directly from the state. A `Sampler` circuit *must* have measurements, or you'll get empty/erroring results.
- **Treating phase as classical probability.** A relative phase (e.g. between $|+\rangle$ and $|-\rangle$) doesn't show up in raw Z-basis counts — you need a basis change (Section 6) or an X/Y observable (Section 8) to detect it.
- **Trying to get a `Statevector` from measurement counts.** Counts are a lossy, probabilistic summary; you cannot reconstruct amplitudes or phase from counts alone in general.
- **Misreading Pauli string qubit order.** `"ZI"` acts $Z$ on qubit 0 (rightmost), identity on qubit 1 — same little-endian convention as bitstrings.
- **Using deprecated primitives.** Do not use `qiskit.primitives.Sampler`/`Estimator` (V1, now removed from current `qiskit-ibm-runtime`) — always use `StatevectorSampler`/`StatevectorEstimator` (local) or `SamplerV2`/`EstimatorV2` (Runtime, Notebook 4).